In [ ]:
import os
import sys
import json
import time
import random
import signal
import shutil
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("/content/mnist_hpo_demo")
CONFIG_DIR = ROOT / "configs"
STATUS_DIR = ROOT / "status"
RESULT_DIR = ROOT / "results"
LOG_DIR = ROOT / "logs"
DATA_DIR = ROOT / "data"

for d in [CONFIG_DIR, STATUS_DIR, RESULT_DIR, LOG_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Working directory:", ROOT)

Working directory: /content/mnist_hpo_demo


In [ ]:
import torchvision
from torchvision import transforms

_ = torchvision.datasets.MNIST(
    root=str(DATA_DIR),
    train=True,
    download=True,
    transform=transforms.ToTensor(),
)

_ = torchvision.datasets.MNIST(
    root=str(DATA_DIR),
    train=False,
    download=True,
    transform=transforms.ToTensor(),
)

print("MNIST downloaded.")

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 497kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.58MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.67MB/s]

MNIST downloaded.


The following cell is the training script for one trial, which we save to `/content/mnist_hpo_demo/mnist_worker.py`

In [ ]:
%%writefile /content/mnist_hpo_demo/mnist_worker.py
import os
import sys
import json
import time
import math
import traceback
import argparse
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms


def save_json_atomic(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


class MLP(nn.Module):
    def __init__(self, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 10),
        )

    def forward(self, x):
        return self.net(x)


class SmallCNN(nn.Module):
    def __init__(self, channels=16, dropout=0.1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 28 -> 14
            nn.Conv2d(channels, 2 * channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 14 -> 7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(2 * channels * 7 * 7, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def make_model(cfg):
    if cfg["model_type"] == "mlp":
        return MLP(hidden_dim=cfg["hidden_dim"], dropout=cfg["dropout"])
    elif cfg["model_type"] == "cnn":
        return SmallCNN(channels=cfg["channels"], dropout=cfg["dropout"])
    else:
        raise ValueError(f"Unknown model_type: {cfg['model_type']}")


def make_loaders(data_dir, batch_size, split_seed):
    transform = transforms.ToTensor()

    full_train = torchvision.datasets.MNIST(
        root=data_dir,
        train=True,
        download=False,
        transform=transform,
    )

    # Keep the demo fast.
    train_size = 10000
    val_size = 2000
    rest_size = len(full_train) - train_size - val_size

    train_subset, val_subset, _ = random_split(
        full_train,
        [train_size, val_size, rest_size],
        generator=torch.Generator().manual_seed(split_seed),
    )

    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_subset,
        batch_size=512,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    return train_loader, val_loader


def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    loss_fn = nn.CrossEntropyLoss()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = loss_fn(logits, y)
            loss_sum += loss.item() * x.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += x.size(0)

    return {
        "loss": loss_sum / total,
        "acc": correct / total,
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--status", required=True)
    parser.add_argument("--result", required=True)
    parser.add_argument("--data-dir", required=True)
    args = parser.parse_args()

    with open(args.config, "r") as f:
        cfg = json.load(f)

    trial_id = cfg["trial_id"]
    start_time = time.time()

    def write_status(extra):
        payload = {
            "trial_id": trial_id,
            "config": cfg,
            "elapsed_s": round(time.time() - start_time, 2),
            **extra,
        }
        save_json_atomic(args.status, payload)

    try:
        # Keep CPU contention low when running multiple workers.
        torch.set_num_threads(1)

        seed = cfg["seed"]
        torch.manual_seed(seed)

        device = "cuda" if torch.cuda.is_available() else "cpu"

        if device == "cuda":
            frac = cfg.get("gpu_fraction", 0.45)
            torch.cuda.memory.set_per_process_memory_fraction(frac, device=0)

        train_loader, val_loader = make_loaders(
            data_dir=args.data_dir,
            batch_size=cfg["batch_size"],
            split_seed=seed,
        )

        model = make_model(cfg).to(device)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cfg["lr"],
            weight_decay=cfg["weight_decay"],
        )
        loss_fn = nn.CrossEntropyLoss()

        best_val_acc = -1.0
        history = []

        write_status({
            "state": "running",
            "epoch": 0,
            "best_val_acc": best_val_acc,
            "history": history,
        })

        for epoch in range(1, cfg["epochs"] + 1):
            model.train()
            train_loss_sum = 0.0
            train_total = 0

            for x, y in train_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                logits = model(x)
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()

                train_loss_sum += loss.item() * x.size(0)
                train_total += x.size(0)

            train_loss = train_loss_sum / train_total
            val_metrics = evaluate(model, val_loader, device)
            best_val_acc = max(best_val_acc, val_metrics["acc"])

            epoch_info = {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_metrics["loss"],
                "val_acc": val_metrics["acc"],
                "best_val_acc": best_val_acc,
            }
            history.append(epoch_info)

            print(
                f"[trial {trial_id}] "
                f"epoch={epoch}/{cfg['epochs']} "
                f"train_loss={train_loss:.4f} "
                f"val_loss={val_metrics['loss']:.4f} "
                f"val_acc={val_metrics['acc']:.4f} "
                f"best={best_val_acc:.4f}",
                flush=True,
            )

            write_status({
                "state": "running",
                "epoch": epoch,
                "best_val_acc": best_val_acc,
                "history": history,
            })

        result = {
            "trial_id": trial_id,
            "state": "finished",
            "score": best_val_acc,
            "config": cfg,
            "history": history,
            "elapsed_s": round(time.time() - start_time, 2),
        }
        save_json_atomic(args.result, result)
        write_status(result)

        del model, optimizer, train_loader, val_loader
        if device == "cuda":
            torch.cuda.empty_cache()

    except Exception as e:
        err = {
            "trial_id": trial_id,
            "state": "error",
            "config": cfg,
            "error": repr(e),
            "traceback": traceback.format_exc(),
            "elapsed_s": round(time.time() - start_time, 2),
        }
        save_json_atomic(args.result, err)
        write_status(err)
        print(err["traceback"], flush=True)
        raise


if __name__ == "__main__":
    main()

Overwriting /content/mnist_hpo_demo/mnist_worker.py


Launcher script: keeps up to max_parallel workers alive in the background

In [ ]:
%%writefile /content/mnist_hpo_demo/mnist_launcher.py
import os
import sys
import json
import time
import signal
import argparse
import subprocess
from pathlib import Path


def save_json_atomic(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--root", required=True)
    parser.add_argument("--max-parallel", type=int, default=2)
    args = parser.parse_args()

    root = Path(args.root)
    config_dir = root / "configs"
    status_dir = root / "status"
    result_dir = root / "results"
    log_dir = root / "logs"
    data_dir = root / "data"
    state_path = root / "launcher_state.json"
    stop_flag = root / "STOP"

    worker_py = root / "mnist_worker.py"

    config_paths = sorted(config_dir.glob("trial_*.json"))
    pending = list(config_paths)
    running = {}
    finished = []

    def write_state(state):
        save_json_atomic(state_path, state)

    write_state({
        "state": "running",
        "pending": [p.name for p in pending],
        "running": {},
        "finished": [],
    })

    try:
        while pending or running:
            if stop_flag.exists():
                for trial_id, meta in list(running.items()):
                    try:
                        meta["proc"].terminate()
                    except Exception:
                        pass
                write_state({
                    "state": "stopped",
                    "pending": [p.name for p in pending],
                    "running": {
                        str(k): {"pid": v["proc"].pid, "log": v["log_path"].name}
                        for k, v in running.items()
                    },
                    "finished": finished,
                })
                return

            while pending and len(running) < args.max_parallel:
                cfg_path = pending.pop(0)
                with open(cfg_path, "r") as f:
                    cfg = json.load(f)
                trial_id = cfg["trial_id"]

                status_path = status_dir / f"trial_{trial_id:03d}.json"
                result_path = result_dir / f"trial_{trial_id:03d}.json"
                log_path = log_dir / f"trial_{trial_id:03d}.log"

                log_file = open(log_path, "w")
                env = os.environ.copy()

                proc = subprocess.Popen(
                    [
                        sys.executable,
                        str(worker_py),
                        "--config", str(cfg_path),
                        "--status", str(status_path),
                        "--result", str(result_path),
                        "--data-dir", str(data_dir),
                    ],
                    stdout=log_file,
                    stderr=subprocess.STDOUT,
                    env=env,
                )

                running[trial_id] = {
                    "proc": proc,
                    "log_file": log_file,
                    "log_path": log_path,
                    "cfg_path": cfg_path,
                }

            done_ids = []
            for trial_id, meta in running.items():
                ret = meta["proc"].poll()
                if ret is not None:
                    meta["log_file"].close()
                    finished.append(trial_id)
                    done_ids.append(trial_id)

            for trial_id in done_ids:
                del running[trial_id]

            write_state({
                "state": "running",
                "pending": [p.name for p in pending],
                "running": {
                    str(k): {"pid": v["proc"].pid, "log": v["log_path"].name}
                    for k, v in running.items()
                },
                "finished": finished,
            })

            time.sleep(2.0)

        write_state({
            "state": "finished",
            "pending": [],
            "running": {},
            "finished": finished,
        })

    finally:
        for trial_id, meta in list(running.items()):
            try:
                meta["log_file"].close()
            except Exception:
                pass


if __name__ == "__main__":
    main()

Writing /content/mnist_hpo_demo/mnist_launcher.py


Some helpers:

In [ ]:
import os
import json
import time
import random
import signal
import subprocess
from pathlib import Path

LAUNCHER_PROC = None

def reset_demo_dir():
    for d in [CONFIG_DIR, STATUS_DIR, RESULT_DIR, LOG_DIR]:
        if d.exists():
            for p in d.glob("*"):
                p.unlink()
    stop_flag = ROOT / "STOP"
    if stop_flag.exists():
        stop_flag.unlink()
    state_path = ROOT / "launcher_state.json"
    if state_path.exists():
        state_path.unlink()
    print("Cleared old configs/status/results/logs.")


def make_trial_configs(n_trials=8, seed=123):
    rng = random.Random(seed)

    model_types = ["mlp", "cnn"]
    lrs = [1e-4, 3e-4, 1e-3]
    batch_sizes = [64, 128, 256]
    dropouts = [0.0, 0.1, 0.2]
    hidden_dims = [128, 256, 512]
    channels_list = [8, 16, 32]

    for trial_id in range(n_trials):
        model_type = rng.choice(model_types)
        cfg = {
            "trial_id": trial_id,
            "seed": 1000 + trial_id,
            "model_type": model_type,
            "lr": rng.choice(lrs),
            "batch_size": rng.choice(batch_sizes),
            "dropout": rng.choice(dropouts),
            "weight_decay": 1e-4,
            "epochs": 3,
            "gpu_fraction": 0.45,   # good starting point for 2 workers
        }

        if model_type == "mlp":
            cfg["hidden_dim"] = rng.choice(hidden_dims)
            cfg["channels"] = None
        else:
            cfg["hidden_dim"] = None
            cfg["channels"] = rng.choice(channels_list)

        path = CONFIG_DIR / f"trial_{trial_id:03d}.json"
        with open(path, "w") as f:
            json.dump(cfg, f, indent=2)

    print(f"Wrote {n_trials} trial configs to {CONFIG_DIR}")


def start_hpo(max_parallel=2):
    global LAUNCHER_PROC

    stop_flag = ROOT / "STOP"
    if stop_flag.exists():
        stop_flag.unlink()

    launcher_py = ROOT / "mnist_launcher.py"

    LAUNCHER_PROC = subprocess.Popen(
        [
            sys.executable,
            str(launcher_py),
            "--root", str(ROOT),
            "--max-parallel", str(max_parallel),
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print(f"Launcher started with PID {LAUNCHER_PROC.pid}")


def read_json(path):
    if not Path(path).exists():
        return None
    try:
        with open(path, "r") as f:
            return json.load(f)
    except Exception:
        return None


def refresh_hpo():
    state = read_json(ROOT / "launcher_state.json")
    if state is None:
        print("No launcher_state.json yet.")
        return None

    rows = []
    for cfg_path in sorted(CONFIG_DIR.glob("trial_*.json")):
        with open(cfg_path, "r") as f:
            cfg = json.load(f)

        trial_id = cfg["trial_id"]
        status = read_json(STATUS_DIR / f"trial_{trial_id:03d}.json")
        result = read_json(RESULT_DIR / f"trial_{trial_id:03d}.json")

        row = {
            "trial_id": trial_id,
            "model": cfg["model_type"],
            "lr": cfg["lr"],
            "batch": cfg["batch_size"],
            "dropout": cfg["dropout"],
            "hidden_dim": cfg["hidden_dim"],
            "channels": cfg["channels"],
            "state": "pending",
            "epoch": None,
            "best_val_acc": None,
            "elapsed_s": None,
        }

        if status is not None:
            row["state"] = status.get("state", "unknown")
            row["epoch"] = status.get("epoch")
            row["best_val_acc"] = status.get("best_val_acc", status.get("score"))
            row["elapsed_s"] = status.get("elapsed_s")

        if result is not None and result.get("state") == "finished":
            row["state"] = "finished"
            row["best_val_acc"] = result.get("score")
            row["elapsed_s"] = result.get("elapsed_s")

        if result is not None and result.get("state") == "error":
            row["state"] = "error"

        rows.append(row)

    df = pd.DataFrame(rows).sort_values(
        by=["state", "best_val_acc"],
        ascending=[True, False],
        na_position="last",
    )

    print("Launcher state:", state["state"])
    print("Pending:", len(state["pending"]), "| Running:", len(state["running"]), "| Finished:", len(state["finished"]))
    display(df)

    finished = [r for r in rows if r["state"] == "finished" and r["best_val_acc"] is not None]
    if finished:
        best = max(finished, key=lambda r: r["best_val_acc"])
        print("\nBest finished trial so far:")
        print(best)

    return df


def tail_log(trial_id, n=30):
    path = LOG_DIR / f"trial_{trial_id:03d}.log"
    if not path.exists():
        print(f"No log for trial {trial_id}")
        return
    with open(path, "r") as f:
        lines = f.readlines()
    print("".join(lines[-n:]))


def stop_hpo():
    stop_flag = ROOT / "STOP"
    stop_flag.write_text("stop\n")
    print("Stop flag written. The launcher will stop and terminate running workers shortly.")

Create training config and start the background run

In [ ]:
reset_demo_dir()
make_trial_configs(n_trials=8, seed=123)
start_hpo(max_parallel=2)

Cleared old configs/status/results/logs.
Wrote 8 trial configs to /content/mnist_hpo_demo/configs
Launcher started with PID 7937


Dashboard; run whenever you want

In [ ]:
refresh_hpo()

Launcher state: finished
Pending: 0 | Running: 0 | Finished: 8


,trial_id,model,lr,batch,dropout,hidden_dim,channels,state,epoch,best_val_acc,elapsed_s
3,3,cnn,0.0010,128,0.2,NaN,8.0,finished,None,0.9275,5.48
0,0,mlp,0.0003,64,0.1,256.0,NaN,finished,None,0.9065,5.38
5,5,cnn,0.0001,64,0.1,NaN,32.0,finished,None,0.8995,5.58
2,2,cnn,0.0003,64,0.0,NaN,8.0,finished,None,0.8975,5.94
6,6,cnn,0.0001,64,0.0,NaN,32.0,finished,None,0.8955,5.56
4,4,mlp,0.0001,128,0.0,512.0,NaN,finished,None,0.8750,4.51
7,7,mlp,0.0001,64,0.1,256.0,NaN,finished,None,0.8735,5.13
1,1,mlp,0.0001,128,0.2,512.0,NaN,finished,None,0.8510,5.34



Best finished trial so far:
{'trial_id': 3, 'model': 'cnn', 'lr': 0.001, 'batch': 128, 'dropout': 0.2, 'hidden_dim': None, 'channels': 8, 'state': 'finished', 'epoch': None, 'best_val_acc': 0.9275, 'elapsed_s': 5.48}


,trial_id,model,lr,batch,dropout,hidden_dim,channels,state,epoch,best_val_acc,elapsed_s
3,3,cnn,0.0010,128,0.2,NaN,8.0,finished,None,0.9275,5.48
0,0,mlp,0.0003,64,0.1,256.0,NaN,finished,None,0.9065,5.38
5,5,cnn,0.0001,64,0.1,NaN,32.0,finished,None,0.8995,5.58
2,2,cnn,0.0003,64,0.0,NaN,8.0,finished,None,0.8975,5.94
6,6,cnn,0.0001,64,0.0,NaN,32.0,finished,None,0.8955,5.56
4,4,mlp,0.0001,128,0.0,512.0,NaN,finished,None,0.8750,4.51
7,7,mlp,0.0001,64,0.1,256.0,NaN,finished,None,0.8735,5.13
1,1,mlp,0.0001,128,0.2,512.0,NaN,finished,None,0.8510,5.34


In [ ]:
tail_log(trial_id=0, n=50)

[trial 0] epoch=1/3 train_loss=1.1362 val_loss=0.5492 val_acc=0.8600 best=0.8600
[trial 0] epoch=2/3 train_loss=0.4558 val_loss=0.3861 val_acc=0.8950 best=0.8950
[trial 0] epoch=3/3 train_loss=0.3574 val_loss=0.3417 val_acc=0.9065 best=0.9065

